# 1.1.9 Error and Exception Handling

Custom exceptions, retry decorator, context managers, logging — applied to a data pipeline.

In [ ]:
import logging, urllib.request
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger('demo')

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA/'titanic.csv'
if not dest.exists():
    try:
        urllib.request.urlretrieve('https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv', dest)
        logger.info(f'Downloaded {dest.name}')
    except Exception as e:
        logger.warning(f'Download failed: {e} — using synthetic data')
        dest.write_text('PassengerId,Survived,Pclass,Age,Fare\n' + '\n'.join(f'{i},{i%2},{(i%3)+1},{20+i},{10+i}' for i in range(1,30)))
print('Ready:', dest.stat().st_size, 'bytes')

In [ ]:
# Custom exception hierarchy
class PipelineError(Exception): pass
class DataValidationError(PipelineError):
    def __init__(self, field, msg):
        super().__init__(f"Validation failed for '{field}': {msg}")
        self.field = field
class ModelTrainingError(PipelineError): pass

try:
    raise DataValidationError('Age', 'non-numeric value "N/A"')
except DataValidationError as e:
    print(f'Caught {type(e).__name__}: {e}  (field={e.field})')

In [ ]:
# Retry decorator
import functools, time, random
def retry(max_attempts=3, base_delay=0.5):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts+1):
                try: return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts: raise
                    print(f'Attempt {attempt} failed: {e}. Retrying…')
                    time.sleep(base_delay * 2**(attempt-1) * 0.01)  # tiny in demo
        return wrapper
    return decorator

@retry(max_attempts=3)
def flaky_call(fail_until):
    if random.random() < fail_until: raise ConnectionError('timeout')
    return 'success'

random.seed(42)
try: print('Result:', flaky_call(0.9))
except ConnectionError: print('All retries exhausted')

In [ ]:
# Context manager
import csv, time
class Timer:
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        self.elapsed = time.perf_counter() - self.t0
        return False

with Timer() as t:
    with open(DATA/'titanic.csv', newline='') as f:
        rows = list(csv.DictReader(f))
print(f'Loaded {len(rows)} rows in {t.elapsed*1000:.2f} ms')

In [ ]:
# except/else/finally
try:
    bad = int('not_a_number')
except ValueError as e:
    print('ValueError caught:', e)
else:
    print('No error — this only runs on success')
finally:
    print('finally always runs')